In [1]:
version = "v2.3.1"
revision_date = "20240211"

In [2]:
# update this string!
student_name = "Daniel"

In [3]:
print(f"Notebook version {version}")

if student_name == "First Last":
    raise Exception("Please update your name in the 'student_name' variable at the top of this notebook.")

print(f"Student name: {student_name}")

Notebook version v2.3.1
Student name: Daniel


# **Course Minimum Score Notice**
- Student must achieve a _**Passing**_ score on **ALL** course assignments and quizzes to receive credit for this course.  
- What defines a _**Passing**_ score?
  - Students must attain a **Raw Score** of not less than 80.0% on **All** assignments and quizzes.  
  - The **Raw Score** is that score achieved prior to the _final_ application of incurred late penalties. 

<a id='toc'></a>
# Table of Contents
- **[Assignment 4 Description](#Topic0)**
  - [Task 1 - Import CSV Sensor Data file](#t1)
  - [Task 2 - Standard train_test_split](#t2)
  - [Task 3 - Build a baseline Tree model](#t3)
  - [Task 4 - Baseline Tree model questions](#t4)
  - [Task 5a - A custom train_test_split function](#t5a)
  - [Task 5b - Rerun the baseline_model](#t5b)
  - [Task 5c - Custom train/test split questions](#t5c)
  - [Task 6 - Mulit-Class Confusion Matrix](#t6)
    - [Visualizing the Confusion Matrix](#vcm)
  - [Task 7 - Confusion Matrix Understanding](#t7)
  - [Task 8 - Feature Importance, part 1](#t8)
  - [Task 9 - Feature Importance, part 2](#t9)  
    - [Scores Plot](#scoresplot)
  - [Task 10 - Final project](#t10)
    - [Task 10 Autograder Scoring](#t10ag)

In [ ]:
# Either of the following is no longer
# necessary for matplotlib in notebooks.
# The import statement has you covered!

# %matplotlib notebook
# %matplotlib inline

<a id='Topic0'></a>
# Assignment 4 - Tree-based classification & Synthesis Project

### Physiological Sensor Data Analysis (100 points)
This synthesis project is based on a dataset of physiological sensor measurements collected from Smartphone based sensors. The original research sought to determine the particular activity of the subject based on the physiological measurements obtained from wearables and a SmartPhone. The physiological measurements were used to depict the test subject in one of four activities as follows:  
   - neutral
   - emotional
   - mental
   - physical  
 
Your task will be to produce a model that, based on a limited number of features, returns the best possible estimate of the activity being performed by the test subject. While the original analysis utilized more advanced Machine Learning methods, we will concentrate on the supervised learning methods covered in this course.  

The sensor dataset consists of 4480 rows, each with the subject ID, the activity label and 533 measurement features! Each of the 40 test volunteers were subjected to a series of 28 data collection events for each of the four activity types presented above. As you explore this data, you will find that the features are arranged by a particular measurement mode, each consisting of similar statistical values.  
Before we get started, it will be necessary to ingest and prepare our data for training and testing purposes.  

**Notes**  
 - Any available random_state or seed values should be initialized with an integer value of 42.
 - Some standard package imports have been provided below.
 - Additional import deemed necessary for your analysis can be added in the cell following.  
 
 <a href='#toc'>TOC</a>

<a id='library_imports'></a>
#### library imports

In [4]:
# Suppress all warnings only when absolutely necessary
# Warnings are in place for a reason!
import warnings

# warnings.filterwarnings('ignore')
# warnings.simplefilter('ignore')

In [5]:
# useful python standard libraries
import itertools
import math
import random
from typing import Callable, List, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from mads.lib.path import assets

In [6]:
# path/filename to dataset
sensor_data = assets.find("sensor_data.csv")

# We will use this variable name multiple times
base_feature_selector = "_mad_"

np.set_printoptions(precision=4)

In [7]:
## Additional imports can be inlcuded here

# import <package>
# import <package> as <alias>
# from <package> import <module>
# from <package>.<module> import <class/function>

<a id='t1'></a>
## Task 1 - Import CSV Sensor Data file (2 points).
 - The first order of business is to read the data file, '_sensor_data.csv_ ' using 'assets.find("sensor_data.csv")'.
 - Your function should accept zero arguments and return a Pandas DataFrame.
   - **See the type-hinted function definition below for details**
 - Be aware that the raw activity labels are in a form that may or may not work with your chosen model(s). Therefore, reassign the activity labels using the following mapping:  
    - **'neutral' == 1**
    - **'emotional' == 2**
    - **'mental' == 3**
    - **'physical' == 4**  
    
<a href='#toc'>TOC</a>

In [10]:
# hidden autograder codeblock
task_id = "1"

In [75]:
def get_sensor_data() -> pd.DataFrame:
    """This would be the location of the doc_string"""

    df = None

    # YOUR CODE HERE

    df = pd.read_csv(assets.find("sensor_data.csv"))

    # 2. Define the mapping
    label_mapping = {
        'neutral': 1,
        'emotional': 2,
        'mental': 3,
        'physical': 4
    }
    
    # 3. Map the column 'Activity_Label' (do NOT rename it yet)
    df['Activity_Label'] = df['Activity_Label'].map(label_mapping)

    return df

In [76]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# get_sensor_data().head()

In [77]:
# Autograder tests
print(f"Task {task_id} - AG tests")

stu_ans = get_sensor_data()

assert isinstance(
    stu_ans, pd.DataFrame
), f"Task {task_id}: Your get_sensor_data function must return a Pandas DataFrame."

assert isinstance(stu_ans.iloc[0, 2], float) or isinstance(stu_ans.iloc[0, 2], np.floating), (
    f"Task {task_id}: The dtype of the first row, third column, is incorrect."
)

# Some hidden tests
del stu_ans

Task 5b - AG tests


<a id='t2'></a>
## Task 2 - Standard train_test_split (3 points).
- Our first exercise will be to produce a SciKit-Learn standard train/test split of the sensor data dataframe.
  - In the following cell, complete the function `std_train_test_split()`.
- The function should accept the following arguments:
  - `dataframe` as produced by get_sensor_data()
  - `test_size` value, may be float or integer (defaults to 0.2).
  - `random_state` integer value (defaults to 42).
- The function will return a tuple of the standard X_train, X_test, y_train, y_test pandas objects.
  - **See the type-hinted function definition below for details**
- The internal [train_test_split()](https://scikit-learn.org/1.2/modules/generated/sklearn.model_selection.train_test_split.html#sklearn.model_selection.train_test_split) function will necessarily incorporate the **stratify** parameter.

<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "2"

In [78]:
def std_train_test_split(
    df: pd.DataFrame, *, test_size: Union[float, int] = 0.2, random_state: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """This would be the location of the doc_string"""

    X_train, X_test, y_train, y_test = (None,) * 4

    # YOUR CODE HERE

    # We must use 'Activity_Label' to match Task 1
    X = df.drop(columns=['Activity_Label'])
    y = df['Activity_Label']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    #raise NotImplementedError()

    return X_train, X_test, y_train, y_test

In [79]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# X_train, X_test, y_train, y_test = std_train_test_split(get_sensor_data(), test_size=0.2, random_state=42)

In [80]:
# Autograder tests
print(f"Task {task_id} - AG tests")

test_size: Union[float, int] = 0.2
random_state: int = 42
df: pd.DataFrame = get_sensor_data()
stu_ans = std_train_test_split(df, test_size=test_size, random_state=random_state)

# print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(stu_ans[0], pd.DataFrame), f"Task {task_id}: X_train should be a pd.DataFrame"

assert isinstance(stu_ans[2], pd.Series), f"Task {task_id}: y_train should be a pd.Series"

# Some hidden tests

del test_size, random_state, df
del stu_ans

Task 5b - AG tests


<a id='t3'></a>
## Task 3 - Build a Baseline Tree Model (5 Points).

- Complete the following function that will establish a baseline score.
- This baseline function will have as parameters:
  - `callable` (function without parentheses) referencing your data splitter function
  - `test_split` as a float or integer value (defaulted to a value of 0.2)
  - `random_state`, an integer value (defaulted to a value of 42).
  - Return a tuple consisting of the list of extracted features and score.
    - **See the type-hinted function definition below for details**
- This function should retrieve and split the sensor data using your previously constructed functions.
  - Reference your std_train_test_split() function definition when setting up to execute the callable within this function.
  - Using a _for loop_ or [list comprehension](https://www.w3schools.com/python/python_lists_comprehension.asp), extract a list of feature names that includes the substring **base_feature_selector** defined in the **[library imports](#library_imports)** section above.
- Create a [DecisionTreeClassifier](https://scikit-learn.org/1.2/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier) with default hyperparameters and  random_state value.
  - Train this model on the features as extracted above.  
  - Using the classifier's [score method](https://scikit-learn.org/1.2/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier.score), score the model using X_test and the previously extracted subset of features.

<a href='#toc'>TOC</a>

In [20]:
# hidden autograder codeblock
task_id = "3"

In [81]:
from sklearn.tree import DecisionTreeClassifier

def baseline_model(
    tts: Callable, *, test_size: Union[float, int] = 0.2, random_state: int = 42
) -> tuple[list[str, ...], float]:
    """This would be the location of the doc_string"""

    features, score = None, None

    # YOUR CODE HERE

    # 1. Get the data (calls Task 1)
    df = get_sensor_data()
    
    # 2. Split the data (calls Task 2)
    X_train, X_test, y_train, y_test = tts(df, test_size=test_size, random_state=random_state)

    # 3. Extract feature names containing "_mad_"
    features = [col for col in X_train.columns if "_mad_" in col]

    # 4. Train the model
    clf = DecisionTreeClassifier(random_state=random_state)
    clf.fit(X_train[features], y_train)

    # 5. Score the model
    score = clf.score(X_test[features], y_test)
    
    #raise NotImplementedError()

    return (features, score)

In [82]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# baseline_model(std_train_test_split, test_size=0.2, random_state=42)

In [83]:
# Autograder tests
print(f"Task {task_id} - AG tests")

test_size: Union[float, int] = 0.2
random_state: int = 42

stu_ans = baseline_model(std_train_test_split, test_size=test_size, random_state=random_state)
print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(stu_ans, tuple), "Task 3: Your function should return a tuple."

assert isinstance(stu_ans[0], list), f"Task {task_id}: Tuple element zero should be a list object."

assert isinstance(stu_ans[1], float) or isinstance(stu_ans[1], np.floating), (
    f"Task {task_id}: Tuple element one should be a Python float or numpy float, but got {type(stu_ans[1])}."
)
# Some hidden tests

del test_size, random_state
del stu_ans

Task 5b - AG tests
Task 5b - your answer:
(['ECG_original_mad_13', 'ECG_RR_window_mad_27', 'ECG_amplitude_RR_mad_41', 'ECG_HR_min_div_mad_55', 'ECG_hrv_mad_79', 'ECG_PSD_mad_93', 'ECG_p_VFL_mad_107', 'ECG_p_LF_mad_121', 'ECG_p_MF_mad_135', 'ECG_p_HF_mad_149', 'ECG_p_total_LF_mad_163', 'IT_Original_mad_187', 'IT_LF_mad_202', 'IT_RF_mad_217', 'IT_BRV_mad_233', 'IT_PSD_mad_247', 'IT_VLF_mad_261', 'IT_LF_mad_275', 'IT_MF_mad_289', 'IT_HF_mad_303', 'IT_p_Total_mad_317', 'EDA_Original_mad_338', 'EDA_processed_mad_352', 'EDA_Filt1_mad_366', 'EDA_Filt2_mad_380', 'EDA_Original_mad_442', 'EDA_processed_mad_456', 'EDA_Filt1_mad_470', 'EDA_Filt2_mad_484'], 0.8069196428571429)


<a id='t4'></a>
## Task 4 - Baseline Tree model questions (3 Points).

1. Does the default accuracy score returned by the model seem reasonable to you; why or why not?
2. What might be the problem with this model or with the data?  

<a href='#toc'>TOC</a>

YOUR ANSWER HERE

1. No, the score does not seem reasonable. While a high accuracy is usually the goal, an accuracy this high on the first try with a "baseline" model using only a fraction of the features (the _mad_ features) is a red flag. In complex physiological sensor data, we expect more "noise" and difficulty in distinguishing between activities like "mental" and "emotional" states.

2. The Issue: The dataset contains multiple rows of data for each Subject_ID. When we used a standard train_test_split in Task 2, the data was shuffled randomly by row. This means that data from "Subject A" performing an activity ended up in both the training and the testing sets.

The Result: The model isn't actually learning how to identify "neutral" vs. "physical" activities; instead, it is learning to recognize the unique physiological "fingerprint" of the specific individuals in the training set. When it sees those same individuals in the test set, it simply "remembers" them rather than generalizing.

<a id='t5a'></a>
## Task 5a- A custom train_test_split function (10 Points).

- Because of the nature of the original experiment's data collection methodology, the standard sklearn train_test_split() method cannot be applied successfully to this dataset.
- The first significant task will be to create a 'custom_train_test_split()' function that will correctly separate train data from test data, given the structure of the data in the sensor dataset.
- Your function should accept three arguments:
  - `pd.DataFrame` a dataframe such as returned by your get_sensor_data() function.
  - `test_size` n integer ___or___ float value that indicates the count or proportion of the **test** data split.
  - `random_state` an integer value
- The function should return of tuple of two DataFrames and 2 Series
  - **See the type-hinted function definition below for details**
- **Note**, your train and test data should **retain** the 'Subject_ID' column!
  - This is **NOT** a requirement to use the 'Subject_ID' in the model.
- Again, any random_state or seed values should be initialized with an integer value of 42.
- In accordance with SciKit-Learn standards, split count determinations should round up.
- Helpful Libraries and functions:
  - [numpy.random](https://numpy.org/doc/1.24/reference/random/)
    - [numpy.random.default_rng()](https://numpy.org/doc/1.24/reference/random/generator.html)
    - [numpy.random.default_rng().choice()](https://numpy.org/doc/1.24/reference/random/generated/numpy.random.Generator.choice.html)
  - Python [math](https://docs.python.org/3/library/math.html)  
    - math.ceil()
- Questions to keep in mind while creating this function:
  - How might this function accept and use an integer or float value for the purpose of dividing the dataset?
  - How do I ensure consistent random selection for reproducibility?
  - Most importantly, how do I split this data to avoid one of the more devastating issues in machine learning?  

<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "5a"

In [28]:
import numpy as np
import math

def custom_train_test_split(
    df: pd.DataFrame, test_size: Union[float, int] = 0.2, random_state: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """This would be the location of the doc_string"""

    X_train, X_test, y_train, y_test = (None,) * 4

    # YOUR CODE HERE

    # 1. Get unique Subject IDs
    subjects = df['Subject_ID'].unique()
    n_subjects = len(subjects)

    # 2. Determine how many subjects go into the test set
    if isinstance(test_size, float):
        n_test = math.ceil(n_subjects * test_size)
    else:
        n_test = test_size

    # 3. Randomly select test subjects
    rng = np.random.default_rng(seed=random_state)
    test_subjects = rng.choice(subjects, size=n_test, replace=False)

    # 4. Create masks to separate the data
    test_mask = df['Subject_ID'].isin(test_subjects)
    train_df = df[~test_mask]
    test_df = df[test_mask]

    # 5. Separate features (X) and labels (y)
    # Important: Retain Subject_ID in X as requested, but drop the target 'activity'
    X_train = train_df.drop(columns=['activity'])
    y_train = train_df['activity']
    
    X_test = test_df.drop(columns=['activity'])
    y_test = test_df['activity']

    #raise NotImplementedError()

    return X_train, X_test, y_train, y_test

In [29]:
# use this cell to explore your solution

def test_my_code() -> None:
    """This would be the location of the doc_string"""

    test_size = 5
    random_state = 42
    df = get_sensor_data()

    X_train, X_test, y_train, y_test = custom_train_test_split(
        df, test_size=test_size, random_state=random_state
    )

    print(f"{df.shape[0]}, {X_train.shape[0]}, {X_test.shape[0]}")

    # insert additional code as necessary to complete your testing

    return None


# Remember to comment the following function call before submitting the notebook.

# test_my_code()

In [30]:
# Autograder tests
print(f"Task {task_id} - AG tests")

test_size: Union[float, int] = 5
random_state: int = 42
df = get_sensor_data()

stu_ans = custom_train_test_split(df, test_size, random_state)

# print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(stu_ans[0], pd.DataFrame), f"Task {task_id}: X_train should be a pd.DataFrame"

assert isinstance(stu_ans[2], pd.Series), f"Task {task_id}: y_train should be a pd.Series"

assert (
    stu_ans[2].dtype == "int64" or stu_ans[2].dtype == "int32"
), f"Task {task_id}: y_train should be of dtype 'int64' or 'int32', your y_train dtype: {stu_ans[2].dtype}"

assert (
    stu_ans[3].dtype == "int64" or stu_ans[3].dtype == "int32"
), f"Task {task_id}: y_test should be of dtype 'int64' or 'int32', your y_test dtype: {stu_ans[3].dtype}"

# Some hidden tests

del test_size, random_state, df
del stu_ans

Task 3 - AG tests


<a id='t5b'></a>
## Task 5b - Rerun your baseline_model with one difference.
- The following code cell will execute your previous baseline_model() function, now using your new custom_train_test_split as the callable for the first argument of the function.
- Use the results of this new usage to answer the questions in Task 5c.

<a href='#toc'>TOC</a>

In [31]:
# YOUR CODE HERE

stu_ans = baseline_model(
    custom_train_test_split, 
    test_size=5, 
    random_state=42
)

#raise NotImplementedError()

In [32]:
# Autograder tests
task_id = "5b"
print(f"Task {task_id} - AG tests")

test_size: Union[float, int] = 5
random_state: int = 42

stu_ans = baseline_model(custom_train_test_split, test_size=test_size, random_state=random_state)

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(stu_ans, tuple), "Task 5b: Your function should return a tuple."

assert isinstance(stu_ans[0], list), f"Task {task_id}: Tuple element zero should be a list object."

assert isinstance(stu_ans[1], float) or isinstance(stu_ans[1], np.floating), (
    f"Task {task_id}: Tuple element one should be a Python float or numpy float, but got {type(stu_ans[1])}."
)


del test_size, random_state
del stu_ans

Task 5b - AG tests
Task 5b - your answer:
(['ECG_original_mad_13', 'ECG_RR_window_mad_27', 'ECG_amplitude_RR_mad_41', 'ECG_HR_min_div_mad_55', 'ECG_hrv_mad_79', 'ECG_PSD_mad_93', 'ECG_p_VFL_mad_107', 'ECG_p_LF_mad_121', 'ECG_p_MF_mad_135', 'ECG_p_HF_mad_149', 'ECG_p_total_LF_mad_163', 'IT_Original_mad_187', 'IT_LF_mad_202', 'IT_RF_mad_217', 'IT_BRV_mad_233', 'IT_PSD_mad_247', 'IT_VLF_mad_261', 'IT_LF_mad_275', 'IT_MF_mad_289', 'IT_HF_mad_303', 'IT_p_Total_mad_317', 'EDA_Original_mad_338', 'EDA_processed_mad_352', 'EDA_Filt1_mad_366', 'EDA_Filt2_mad_380', 'EDA_Original_mad_442', 'EDA_processed_mad_456', 'EDA_Filt1_mad_470', 'EDA_Filt2_mad_484'], 0.5857142857142857)


<a id='t5c'></a>
## Task 5c - Custom train/test split questions (2 Points).

1. Is the score of the model that incorporates custom_train_test_split() significantly different from the std_train_test_split() version?  
2. What issue(s) have we eliminated with our new custom_train_test_split() function?  

<a href='#toc'>TOC</a>

YOUR ANSWER HERE

1. Yes, the score is significantly lower. In the standard split, the model likely achieved a very high accuracy (possibly near 90-100%). With the custom split by Subject_ID, the accuracy likely dropped substantially (often to the 40-60% range, depending on the features). This difference indicates that the first model was not actually learning "activities," but was instead memorizing the specific physiological signatures of the individuals present in both sets.

2. We have eliminated Data Leakage (specifically, Group Leakage). By ensuring that data from a single subject never appears in both the training and testing sets, we have forced the model to learn generalizable patterns of movement and physiology rather than subject-specific traits. This provides a much more realistic estimate of how the model will perform on a completely new person (unseen data) in the real world.

<a id='t6'></a>
## Task 6 - Multi-Class Confusion Matrix (5 Points).
- We now want to better understand the relationship of correct and incorrect predictons made by a classification model. A very useful tool for examining a multiclass outcome, such as we have with our sensor dataset, is the [Confusion Matrix](https://scikit-learn.org/1.2/modules/generated/sklearn.metrics.confusion_matrix.html#sklearn.metrics.confusion_matrix)
 - Using the SciKit-Learn [LogisticRegression](https://scikit-learn.org/1.2/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression) model, create a function that returns a Confusion Matrix.
- Your function should accept two arguments.
  - `test_size` value, may be float or integer (defaults to 0.2)
  - `random_state` integer value (defaults to 42).
- Your function should return the confusion_matrix() output, an np.ndarray.
  - **See the type-hinted function definition below for details**
- Set hyperparameter: max_iter=1000
- Any random_state or seed values should be initialized with an integer value of 42.
- Use your previously defined functions to derive your train and test sets.
- Scaling is an important factor in many Machine Learning projects (See section 3.3 in the course textbook). For the current dataset, use the sklearn method StandardScaler() to scale the train and test sets.
 - Using code that you had previously developed, include all features who's name includes the substring defined in 
**base_feature_selector** ([library imports](#library-imports)).

<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "6"

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

def LR_confusion_matrix(test_size: Union[float, int] = 0.2, random_state: int = 42) -> np.ndarray:
    """This would be the location of the doc_string"""

    conf_matrix = None

    # YOUR CODE HERE

    # 1. Get and split the data using our custom subject-based split
    df = get_sensor_data()
    X_train_raw, X_test_raw, y_train, y_test = custom_train_test_split(
        df, test_size=test_size, random_state=random_state
    )

    # 2. Select features containing "_mad_"
    features = [col for col in X_train_raw.columns if "_mad_" in col]
    X_train_sub = X_train_raw[features]
    X_test_sub = X_test_raw[features]

    # 3. Scaling the data
    scaler = StandardScaler()
    # We fit only on training data to avoid further data leakage!
    X_train_scaled = scaler.fit_transform(X_train_sub)
    X_test_scaled = scaler.transform(X_test_sub)

    # 4. Train the Logistic Regression model
    lr = LogisticRegression(max_iter=1000, random_state=random_state)
    lr.fit(X_train_scaled, y_train)

    # 5. Make predictions and generate the confusion matrix
    y_pred = lr.predict(X_test_scaled)
    conf_matrix = confusion_matrix(y_test, y_pred)
    
    #raise NotImplementedError()

    return conf_matrix

In [34]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# LR_confusion_matrix(test_size=0.2, random_state=42)

In [35]:
# Autograder tests
print(f"Task {task_id} - AG tests")

test_size: Union[float, int] = 0.2
random_state: int = 42

stu_ans = LR_confusion_matrix(test_size=test_size, random_state=random_state)

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert isinstance(stu_ans, np.ndarray), f"Task {task_id}: The second tuple element should be an np.ndarray"


del test_size, random_state
del stu_ans

Task 5b - AG tests
Task 5b - your answer:
[[125  71  28   0]
 [ 10 136  59  19]
 [  7  96 104  17]
 [  0  16  33 175]]


<a id='vcm'></a>
## Visualizing the Confusion Matrix
<a href='#toc'>TOC</a>

In [36]:
def plot_confusion(test_size: Union[float, int] = 0.2, random_state: int = 42):
    cm = LR_confusion_matrix(test_size=test_size, random_state=random_state)
    labels = {"neutral": 1, "emotional": 2, "mental": 3, "physical": 4}.keys()
    display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    display_cm.plot()
    plt.show()

In [ ]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# plot_confusion(test_size=0.2, random_state=42)

<a id='t7'></a>
## Task 7 - Confusion Matrix Understanding (5 Points).

 - Answer the following questions concerning the above Confusion Matrix.  
 - Record (hardcode) your answers in the answer variables in the cell below for autograding purposes.  
   - You may manually calculate the answer or make the determinations programmatically.
 - Your answers should use the Python types **int**, **float**, or **str** as appropriate.
 - **See the type-hinted function definition below for details**
 
Q1. What is the number of __Correctly Predicted__ for the _mental_ activity?  
Q2. How many __False Positive__ predictions were made for the _neutral_ activity?  
Q3. How many __False Negative__ predictions were made for the _physical_ activity?  
Q4. What is the __Precision__ Score for the _mental_ activity? (round to three decimal places)  
Q5. What is the __Recall__ Score for the _emotional_ activity? (round to three decimal places)  
Q6. What is the overall __Accuracy__ for the current model? (round to three decimal places)  
Q7. Which activity is most incorrectly labeled as _mental_?  

<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "7"

In [39]:
# Supply your answers to Q1 through Q7
# in variables a1 through a7.


def confusion_matrix_questions() -> tuple[int, int, int, float, float, float, str]:
    a1 = None
    a2 = None
    a3 = None
    a4 = None
    a5 = None
    a6 = None
    a7 = None

    # YOUR CODE HERE

    cm = LR_confusion_matrix(test_size=0.2, random_state=42)
    # Q1: Correctly Predicted for 'mental' (Actual mental, Predicted mental)
    # mental is index 2
    a1 = int(cm[2, 2])
    
    # Q2: False Positives for 'neutral' (Predicted neutral, but actually something else)
    # Sum of column 0 minus the diagonal
    a2 = int(cm[:, 0].sum() - cm[0, 0])
    
    # Q3: False Negatives for 'physical' (Actual physical, but predicted something else)
    # Sum of row 3 minus the diagonal
    a3 = int(cm[3, :].sum() - cm[3, 3])
    
    # Q4: Precision for 'mental' (Correct mental / Total predicted mental)
    # cm[2,2] / sum of column 2
    a4 = float(round(cm[2, 2] / cm[:, 2].sum(), 3))
    
    # Q5: Recall for 'emotional' (Correct emotional / Total actual emotional)
    # cm[1,1] / sum of row 1
    a5 = float(round(cm[1, 1] / cm[1, :].sum(), 3))
    
    # Q6: Overall Accuracy (Sum of diagonal / Sum of everything)
    a6 = float(round(np.trace(cm) / cm.sum(), 3))
    
    # Q7: Which activity is most incorrectly labeled as 'mental'?
    # Look at column 2 (predicted mental). 
    # Exclude the correct one (row 2). Find the max of the remaining rows in that column.
    mental_col = cm[:, 2].copy()
    mental_col[2] = 0 # Ignore correct predictions
    incorrect_idx = np.argmax(mental_col)
    
    mapping = {0: 'neutral', 1: 'emotional', 2: 'mental', 3: 'physical'}
    a7 = str(mapping[incorrect_idx])

    
    #raise NotImplementedError()

    return (a1, a2, a3, a4, a5, a6, a7)

In [40]:
# Autograder tests
print(f"Task {task_id} - AG tests")

stu_ans = confusion_matrix_questions()

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert len(stu_ans) == 7, f"Task {task_id}: Your answer tuple does not contain the correct number of answers."
assert isinstance(stu_ans[0], int), f"Task {task_id}: Answer one should be an integer"
assert isinstance(stu_ans[1], int), f"Task {task_id}: Answer two should be an integer"
assert isinstance(stu_ans[2], int), f"Task {task_id}: Answer three should be an integer"
assert isinstance(stu_ans[3], float), f"Task {task_id}: Answer four should be a float"
assert isinstance(stu_ans[4], float), f"Task {task_id}: Answer five should be a float"
assert isinstance(stu_ans[5], float), f"Task {task_id}: Answer six should be a float"
assert isinstance(stu_ans[6], str), f"Task {task_id}: Answer seven should be a string"


del stu_ans

Task 5b - AG tests
Task 5b - your answer:
(104, 17, 49, 0.464, 0.607, 0.603, 'emotional')


<a id='t8'></a>
## Task 8 - Feature Importance, part 1 (5 Points).
#### Now we want to explore how some models are able to provide additional insight into the features that played a prominent role in the estimation outcome.
 - Produce a function that implements a RandomForestClassifier model, which includes the [**feature_importances_**](https://scikit-learn.org/1.2/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier.feature_importances_) attribute.
- The function accepts three arguments:
  - `top` integer value of X number of features (defaulted toa value of 10).
  - `test_size` value, may be float or integer (defaults to 0.2).
  - `random_state` integer value (defaults to 42).
 - The function should return a tuple of two elements.
   - The first element will be a **sorted** list of tuples in the form **\[('feature_name', importance_value),...\]
     - This list of tuples should be sorted in **descending order** of feature_importance.
   - The second element (test data score) should be an np.float64 value.  
   - **See the type-hinted function definition below for details**
- The classifier should use only two parameters:
  - random_state=42
  - n_jobs=-1
- Use your previously defined functions to derive your train and test sets.
- Using code that you had previously developed, include all features who's name includes the substring defined by **base_feature_selector** ([library imports](#library-imports)).


<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "8"

In [41]:
from sklearn.ensemble import RandomForestClassifier

def get_top_features(
    top: int = 10, test_size: Union[float, int] = 0.2, random_state: int = 42
) -> tuple[list[tuple[str, float], ...], float]:
    """This would be the location of the doc_string"""

    top_x, score = None, None

    # YOUR CODE HERE

    # 1. Get and split the data
    df = get_sensor_data()
    X_train_raw, X_test_raw, y_train, y_test = custom_train_test_split(
        df, test_size=test_size, random_state=random_state
    )

    # 2. Select features containing "_mad_"
    features_list = [col for col in X_train_raw.columns if "_mad_" in col]
    X_train = X_train_raw[features_list]
    X_test = X_test_raw[features_list]

    # 3. Initialize and fit the RandomForestClassifier
    # n_jobs=-1 uses all available processors for faster training
    rf = RandomForestClassifier(random_state=random_state, n_jobs=-1)
    rf.fit(X_train, y_train)

    # 4. Extract and sort feature importances
    importances = rf.feature_importances_
    # Create list of tuples: [('feature_name', 0.123), ...]
    feature_importance_pairs = list(zip(features_list, importances))
    
    # Sort by the importance value (index 1) in descending order
    sorted_features = sorted(feature_importance_pairs, key=lambda x: x[1], reverse=True)
    
    # Slice to get only the requested "top" number
    top_x = sorted_features[:top]

    # 5. Calculate the test score
    score = rf.score(X_test, y_test)
    
    #raise NotImplementedError()

    return top_x, score

In [42]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# get_top_features(top=10, test_size=0.2, random_state=42)

In [43]:
# Autograder tests
print(f"Task {task_id} - AG tests")
top: int = 10
test_size: Union[float, int] = 0.2
random_state: int = 42

stu_ans = get_top_features(top=top, test_size=test_size, random_state=random_state)

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert (
    len(stu_ans) == 2
), f"Task {task_id}: Your answer tuple does not contain the correct number of elements."

assert len(stu_ans[0]) == top, f"Task {task_id}: The default number of top features is not correct."

assert isinstance(stu_ans[0], list), f"Task {task_id}: The first tuple element should be a list"

assert isinstance(stu_ans[1], float) or isinstance(stu_ans[1], np.floating), (
    f"Task {task_id}: The second returned tuple element should be a Python float or numpy float, but got {type(stu_ans[1])}."
)
# Some hidden tests
del top, test_size, random_state
del stu_ans

Task 5b - AG tests
Task 5b - your answer:
([('IT_Original_mad_187', 0.11107459794684481), ('IT_LF_mad_202', 0.09948025602863993), ('IT_RF_mad_217', 0.06931782898890497), ('IT_PSD_mad_247', 0.05450613559564808), ('IT_p_Total_mad_317', 0.05342970354171619), ('IT_MF_mad_289', 0.047916161054636516), ('ECG_amplitude_RR_mad_41', 0.046353499071122795), ('ECG_original_mad_13', 0.043039363633666845), ('EDA_processed_mad_456', 0.03901097273867212), ('IT_HF_mad_303', 0.037432542515542296)], 0.6049107142857143)


<a id='t9'></a>
## Task 9 - Feature Importance, part 2 (10 Points).

#### The value of Feature Importance
- This follow-on task will use the same **RandomForestClassifier** model as in your get_top_features() function.
- The new model will use a portion of the output returned by the get_top_features() function.
- The function accepts three arguments:
  - `top` integer value of X number of features (defaulted toa value of 10).
  - `test_size` value, may be float or integer (defaults to 0.2).
  - `random_state` integer value (defaults to 42).
- Your function should return a list of feature-based test data scores produced by the model.
  - **See the type-hinted function definition below for details**
- Use the previously defined functions to derive your train and test datasets.
- Create a loop that trains your model with an incrementally increasing number of features from the top features list.
  - The first pass will include the topmost important feature, the second pass will include the top two most important features and so on until the final pass of all top X features.

<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "9"

In [44]:
def score_top_features(
    top: int = 10, test_size: Union[float, int] = 0.2, random_state: int = 42
) -> list[float, ...]:
    scores: lista = []

    # YOUR CODE HERE

    # 1. Get the top features using the function from Task 8
    # We only need the names (the first element of the tuples)
    top_feature_tuples, _ = get_top_features(top=top, test_size=test_size, random_state=random_state)
    top_feature_names = [feature[0] for feature in top_feature_tuples]

    # 2. Get the data and split it
    df = get_sensor_data()
    X_train_raw, X_test_raw, y_train, y_test = custom_train_test_split(
        df, test_size=test_size, random_state=random_state
    )

    scores = []

    # 3. Loop through and incrementally add features
    for i in range(1, top + 1):
        # Select the first 'i' features from our sorted list
        current_features = top_feature_names[:i]
        
        # Initialize and train the model
        rf = RandomForestClassifier(random_state=random_state, n_jobs=-1)
        rf.fit(X_train_raw[current_features], y_train)
        
        # Calculate score and add to our list
        current_score = rf.score(X_test_raw[current_features], y_test)
        scores.append(float(current_score))
    
    #raise NotImplementedError()

    return scores

In [45]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# score_top_features(top=10, test_size=0.2, random_state=42)

In [46]:
# Autograder tests
print(f"Task {task_id} - AG tests")
top: int = 10
test_size: Union[float, int] = 0.2
random_state: int = 42

stu_ans = score_top_features(top=top, test_size=test_size, random_state=random_state)

print(f"Task {task_id} - your answer:\n{stu_ans}")

assert len(stu_ans) == top, f"Task {task_id}: Your function does not return the correct number of scores."

assert all(
    isinstance(x, float) or isinstance(x, np.floating) for x in stu_ans
), f"Task {task_id}: One or more of the returned scores is of an incorrect type."


del top, test_size, random_state
del stu_ans

Task 5b - AG tests
Task 5b - your answer:
[0.4341517857142857, 0.47433035714285715, 0.4497767857142857, 0.6015625, 0.6015625, 0.5825892857142857, 0.5725446428571429, 0.5758928571428571, 0.6127232142857143, 0.6037946428571429]


<a id='scoresplot'></a>
## Scores Plot
The plot_scores() function accepts top X features argumnet which defaults to a value of 10. Feel free to change the argument to better see how accuracy is affected by the number of top features used to train the model.  

<a href='#toc'>TOC</a>

In [47]:
def plot_scores(top=10):
    if top > 29:
        top = 29

    top_x_features, score = get_top_features(top)
    scores = score_top_features(top)

    importance_scores = [score[1] for score in top_x_features]
    x_axis = np.arange(1, len(scores) + 1)

    plt.figure(figsize=(8, 6))

    plt.xticks(x_axis)
    plt.ylim([0, max(score, max(importance_scores)) + 0.05])
    plt.xlabel("Number of Top 10 Features Used")
    plt.ylabel("Score Value")
    plt.grid(alpha=0.25)

    plt.plot(x_axis, scores, label="Accuracy Score", c="g", linestyle="solid")

    plt.plot(
        x_axis,
        importance_scores,
        label="Feature Importance Scores",
        c="r",
        linestyle="solid",
    )

    plt.axhline(
        score,
        label=f"All '{base_feature_selector}' features score",
        c="b",
        linestyle="dashed",
    )

    plt.legend()
    plt.tight_layout()

    return

In [48]:
# Plot accuracy vs feature importance
# the plot_scores() function accepts an optional top N parameter.
# Remember to comment the following function call before submitting the notebook.

# plot_scores()

<a id='t10'></a>
## Task 10 - Final project (50 Points).
- The final task for this assignment is open-ended with only a few **constraints**. Using any of the Supervised Machine Learning techniques and models presented in this course, produce a model that produces a best-possible ROC-AUC score. Your **Task 10** points will be based solely on the highest ROC-AUC score achieved. The primary constraint for this task is that you will be able to utilize not more than **10** of the 533 available data features in the training and scoring of your model. A quick calculation of the number of available combinations C(n, r) = C(533, 10) = $\frac{n!}{r!(n-r)!}$ = 4.684e20. That is very large number of possible combinations (the number of permutations is even greater)! Because of the intractability of checking all possible 10-feature combinations, it will be necessary to devise a scheme whereby your algorithm makes a selection of features and scores that selection. Be creative but also efficient in your feature selection process; this may well mean computational efficiency. Attempting to examine too many features at one time can be computationally very expensive! Because multiple feature selection cycles may be necessary, you will also need to develop an efficient method of keeping track of the top model, features, and score.
- Why would we want to limit the number of features?
   1. The creator of the project may want to minimize the number of sensors/measurements required to move the project forward.
   2. The final product will be used on a smartphone where resource consumption is always a concern.
   3. Computational resource availability of the development environment could be constrained due to budget availability.
   4. The development environment may be intentionally constrained to mimic the production environment.
- You will find that even with these limitations, some model choices will still consume significant resources, even to the point of crashing the Python kernel!
- Use your previously defined functions to derive your train and test sets.
- If feasible, expand upon your previously created feature selection code.
- Tree-based models may perform equally well with parameter defaults reduced e.g. n_estimators and max_depth.
- Use the following parameters when establishing your [roc_auc_score](https://scikit-learn.org/1.2/modules/generated/sklearn.metrics.roc_auc_score.html#sklearn.metrics.roc_auc_score) method:
  - average="macro"
  - multi_class="ovr"
- The function should accept the following arguments:
  - `test_size` value, may be float or integer (defaults to 0.2).
  - `random_state` integer value (defaults to 42).
- function return:: tuple: (fit_model, feature_list, roc_auc_score)
  - **See the type-hinted function definition below for details**

<a href='#toc'>TOC</a>

In [ ]:
# hidden autograder codeblock
task_id = "10"

In [65]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np


def activity_model(
    test_size: Union[float, int] = 0.2, random_state: int = 42
) -> Tuple[BaseEstimator, np.ndarray, float]:
    model, features, score = (None,) * 3

    # YOUR CODE HERE

   # 1. Prepare Data
    df = get_sensor_data()
    X_train_raw, X_test_raw, y_train, y_test = custom_train_test_split(
        df, test_size=test_size, random_state=random_state
    )
    
    # Drop Subject_ID for calculation
    X_train_full = X_train_raw.drop(columns=['Subject_ID'], errors='ignore')
    X_test_full = X_test_raw.drop(columns=['Subject_ID'], errors='ignore')

    # 2. Robust Feature Selection using Lasso (L1)
    # Scaling is required for Logistic Regression to work properly
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_full)
    
    # C=0.05 is a strong penalty to narrow down to very few features
    selector = LogisticRegression(
        penalty='l1', solver='liblinear', C=0.1, random_state=random_state, multi_class='ovr'
    )
    selector.fit(X_train_scaled, y_train)
    
    # Get the features with the largest absolute coefficients
    coeffs = np.abs(selector.coef_).sum(axis=0)
    indices = np.argsort(coeffs)[-10:] # Top 10 robust features
    features = X_train_full.columns[indices].to_numpy()

    # 3. Final Model Training
    # We use a Random Forest with balanced class weights to help the AUC
    model = RandomForestClassifier(
        n_estimators=400,
        max_depth=12,
        class_weight='balanced',
        random_state=random_state,
        n_jobs=-1
    )
    model.fit(X_train_full[features], y_train)

    # 4. Calculate ROC-AUC Score
    y_probs = model.predict_proba(X_test_full[features])
    score = roc_auc_score(y_test, y_probs, average="macro", multi_class="ovr")

    
    #raise NotImplementedError()

    return (model, features, score)

In [66]:
# use this cell to explore your solution
# Remember to comment the following function call before submitting the notebook.

# activity_model(test_size=0.2, random_state=42)

<a id='t10ag'></a>
## Task 10 Autograder Scoring
<a href='#toc'>TOC</a>

In [67]:
# Hidden autograder model validation
# test for AUC score >= 0.80
level = 1

print(f"Task {task_id} - AG tests")

test_size: Union[float, int] = 0.2
random_state: int = 42

stu_ans = activity_model(test_size=test_size, random_state=random_state)

print(f"Task {task_id} - your answers:\n{stu_ans[0]}\n\n{stu_ans[1]}\n\nScore: {stu_ans[2]}\n")

assert isinstance(
    stu_ans[0], BaseEstimator
), f"Task {task_id} - The first element of your result should be an estimator inheriting from the sklearn.base.BaseEstimator class."

assert isinstance(stu_ans[1], list) or isinstance(
    stu_ans[1], np.ndarray
), f"Task {task_id} - The second element of your result should be a numpy array or python list of strings."

assert (
    len(stu_ans[1]) <= 10
), f"Task {task_id}: your feature count of {len(stu_ans[1])} is greater than allowed in the assignment definition."

assert isinstance(
    stu_ans[2], float
), f"Task {task_id} - The third element of your result should be a floating point value."


assert (
    stu_ans[2] >= 0.80
), f"Task {task_id}: Your test AUC {stu_ans[2]:.4f} is less than 0.80. You will receive 0 points for this task."

Task 5b - AG tests


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Task 5b - your answers:
RandomForestClassifier(class_weight='balanced', max_depth=12, n_estimators=400,
                       n_jobs=-1, random_state=42)

['EDA_Original_baseline_443' 'IT_RF_baseline_218' 'EDA_Filt2_baseline_485'
 'IT_BR_mean_220' 'ECG_Logstd_60' 'IT_LF_MF_HF_323' 'IT_HF_LF_319'
 'IT_VLF_baseline_262' 'ECG_hrv_geomean(abs)_77' 'IT_BRV_baseline_234']

Score: 0.9155123963647959



In [68]:
# autograder test - AUC >= 0.84
level = 2


assert (
    stu_ans[2] >= 0.840
), f"Task {task_id}: Your test AUC {stu_ans[2]:.4f} is less than 0.84. You will receive 25 points for this task."

In [69]:
# autograder test - AUC >= 0.86
level = 3


assert (
    stu_ans[2] >= 0.860
), f"Task {task_id}: Your test AUC {stu_ans[2]:.4f} is less than 0.86. You will receive 30 points for this task."

In [70]:
# autograder test - AUC >= 0.88
level = 4


assert (
    stu_ans[2] >= 0.880
), f"Task {task_id}: Your test AUC {stu_ans[2]:.4f} is less than 0.88. You will receive 35 points for this task."

In [71]:
# autograder test - AUC >= 0.90
level = 5


assert (
    stu_ans[2] >= 0.900
), f"Task {task_id}: Your test AUC {stu_ans[2]:.4f} is less than 0.900. You will receive 45 points for this task."

del stu_ans

<a href='#toc'>TOC</a>